# Feature Selection

**Topic:** Feature Engineering

In [ ]:
import numpy as np
import pandas as pd
import plotly.graph_objects as go
import ipywidgets as widgets
from ipywidgets import Dropdown, IntSlider, Output, HBox, VBox
from IPython.display import display, clear_output
from sklearn.ensemble import RandomForestRegressor
from sklearn.inspection import permutation_importance
from sklearn.feature_selection import mutual_info_regression
from sklearn.linear_model import Ridge
from sklearn.model_selection import train_test_split, KFold, cross_val_score
from sklearn.metrics import mean_absolute_error
from tkh_utils import PALETTE, FONT, base_layout, load_airbnb


---
## What you'll explore

By the end of this demo you will be able to:

- **Describe** the three families of feature selection: filter, wrapper, and embedded
- **Explain** why a column can look important under one scoring method and unimportant under another
- **Interpret** a before-and-after comparison to see whether dropping the low-scoring columns actually costs you anything

> **Tip:** Try all three methods in the dropdown before changing anything else. With this feature set, all three agree on which columns matter — that agreement is itself worth noticing.

---
## How we got here

In **[06_handling_skewed_features.ipynb](06_handling_skewed_features.ipynb)** you learned that a column's raw scale can work against a model. In **[ml_concepts/10_feature_importance.ipynb](../ml_concepts/10_feature_importance.ipynb)** you learned to measure how much each column actually contributes to a model's predictions.

Feature selection takes that measurement and acts on it: score every column, rank them, and check whether keeping only the top few costs you anything.

---
## Why this matters for data science

Not every column in a dataset carries a signal. In this dataset, `minimum_nights`, `number_of_reviews`, `availability_365`, and `calculated_host_listings_count` each correlate with price at a strength below 0.005 — essentially zero. The three columns that actually carry signal are whether the listing is an entire home (`is_entire_home`), whether it's in Manhattan (`is_manhattan`), and how far it sits from Midtown (`dist_midtown`).

Feature selection is how you'd find this out on a dataset where you don't already know the answer: score every column with a filter, wrapper, or embedded method, rank them, and see what happens when you drop the bottom scorers. Here, it costs nothing measurable — a model trained on all 7 columns and one trained on just the 3 useful ones score within 0.0001 of each other in cross-validated R², and their typical dollar error differs by about 4 cents. The payoff isn't a more accurate model. It's a model with fewer than half the moving parts, for free.

---
## Try it yourself

In [ ]:
_airbnb = load_airbnb()
_airbnb = _airbnb[_airbnb['price'] > 0].copy()

_airbnb['dist_midtown']   = np.sqrt((_airbnb['latitude'] - 40.7580) ** 2 + (_airbnb['longitude'] - (-73.9855)) ** 2)
_airbnb['is_entire_home'] = (_airbnb['room_type'] == 'Entire home/apt').astype(float)
_airbnb['is_manhattan']   = (_airbnb['neighbourhood_group'] == 'Manhattan').astype(float)

_feat_cols = ['dist_midtown', 'is_entire_home', 'is_manhattan',
              'minimum_nights', 'number_of_reviews',
              'availability_365', 'calculated_host_listings_count']

_X = _airbnb[_feat_cols]
_y = np.log1p(_airbnb['price'])
_price = _airbnb['price'].values

_Xtr, _Xva, _ytr, _yva = train_test_split(_X, _y, test_size=0.25, random_state=42)
_, _, _, _yva_price = train_test_split(_X, _price, test_size=0.25, random_state=42)
_kf = KFold(5, shuffle=True, random_state=42)

# Every score below is computed ONCE, here, when the cell runs — not inside render().
# The slider and dropdown only re-slice and re-draw these cached arrays.
_corr_scores = np.array([abs(np.corrcoef(_X[c], _y)[0, 1]) for c in _feat_cols])
_mi_scores = mutual_info_regression(_X, _y, random_state=42)

_rf = RandomForestRegressor(n_estimators=100, random_state=42, n_jobs=-1).fit(_Xtr, _ytr)
_perm_scores = permutation_importance(
    _rf, _Xva, _yva, n_repeats=5, random_state=42, n_jobs=-1).importances_mean

_SCORES = {
    "Correlation": _corr_scores,
    "Mutual information": _mi_scores,
    "Random Forest (permutation importance)": _perm_scores,
}

_r2_all = cross_val_score(Ridge(), _X, _y, cv=_kf, scoring='r2').mean()
_ridge_all = Ridge().fit(_Xtr, _ytr)
_med_all = float(np.median(np.abs(_yva_price - np.expm1(_ridge_all.predict(_Xva)))))

out_select = Output()
caption_select = widgets.HTML()

n_feat_sl = IntSlider(value=3, min=1, max=len(_feat_cols), step=1,
    description="Top N features:", style={"description_width": "110px"},
    layout=widgets.Layout(width="380px"))
method_dd = Dropdown(
    options=list(_SCORES.keys()),
    value="Random Forest (permutation importance)", description="Method:",
    style={"description_width": "70px"}, layout=widgets.Layout(width="340px"))

def render_select(change=None):
    n = n_feat_sl.value
    method = method_dd.value
    scores = _SCORES[method]
    order = np.argsort(scores)[::-1]
    top_feats = [_feat_cols[i] for i in order[:n]]

    r2 = cross_val_score(Ridge(), _X[top_feats], _y, cv=_kf, scoring='r2').mean()
    ridge_top = Ridge().fit(_Xtr[top_feats], _ytr)
    med = float(np.median(np.abs(_yva_price - np.expm1(ridge_top.predict(_Xva[top_feats])))))

    display_order = order[::-1]
    colors = [PALETTE["primary"] if _feat_cols[i] in top_feats else PALETTE["muted"]
              for i in display_order]

    fig = go.Figure(data=go.Bar(
        x=scores[display_order], y=[_feat_cols[i] for i in display_order],
        orientation="h", marker_color=colors,
        text=[f"{v:.3f}" for v in scores[display_order]], textposition="outside",
    ), layout=base_layout(
        title=f"{method} — top {n} selected",
        xaxis_title="Score", yaxis_title="",
    ))
    with out_select:
        clear_output(wait=True)
        fig.show()

    r2_delta = r2 - _r2_all
    med_delta = med - _med_all
    caption_select.value = (
        f"<b>Top {n} by {method}:</b> {', '.join(top_feats)}. "
        f"5-fold CV R² = {r2:.4f} (all 7 columns: {_r2_all:.4f}); "
        f"typical listing error = ${med:.2f} (all 7 columns: ${_med_all:.2f})."
    )

n_feat_sl.observe(render_select, names="value")
method_dd.observe(render_select, names="value")
display(VBox([HBox([n_feat_sl, method_dd]), out_select, caption_select]))
render_select()


---
## What's happening?

There are three families of feature selection methods.

**Filter methods** score each column independently using a statistical measure and keep the top scorers. They're fast and model-agnostic. Correlation and mutual information, both above, are filter methods.

**Wrapper methods** train a model repeatedly on different column subsets and keep the subset with the best performance. Recursive Feature Elimination (RFE) is the most common example. Accurate but slow, since it means training many models.

**Embedded methods** perform selection as part of training a single model. A Random Forest's importances are an example: the model ranks columns by how much they're used to make decisions. Lasso regression is another — it drives the weights of unhelpful columns to exactly zero (see **[supervised/04_ridge_lasso_regression.ipynb](../supervised/04_ridge_lasso_regression.ipynb)**).

**A caution about Random Forest importance:** scikit-learn's built-in `.feature_importances_` (impurity-based) is biased toward columns with many distinct values — on this dataset it ranks `availability_365` (a continuous column with no real signal) above `is_manhattan` (a binary column that matters a lot). `permutation_importance`, used above, doesn't have that bias, because it measures the drop in accuracy when a column is shuffled rather than counting how often the model happened to split on it.

| Method | How it works | Model agnostic? | Handles multicollinearity? | sklearn class |
|--------|-------------|-----------------|--------------------------|---------------|
| Correlation | Pearson r with target | Yes | No | `SelectKBest(f_regression)` |
| Mutual information | Non-linear dependence | Yes | No | `SelectKBest(mutual_info_regression)` |
| RFE | Train + prune iteratively | No | Partially | `RFE` |
| RF importance | Impurity reduction per split — biased toward high-cardinality columns; prefer permutation importance for ranking | No | Partially | `RandomForestRegressor.feature_importances_` / `permutation_importance` |
| Lasso | L1 penalty drives weights to 0 | No | Yes | `Lasso` |

### Measured on this dataset

All three methods above agree: `is_entire_home`, `is_manhattan`, and `dist_midtown` are the top 3 of 7. The other four score near zero on every method.

| Feature set | CV R² | MAE | Median error |
|---|---|---|---|
| All 7 columns | 0.3657 | $75.36 | $45.18 |
| Top 3 (useful only) | 0.3658 | $75.36 | $45.14 |

Dropping the 4 useless columns doesn't cost any measurable accuracy — the top-3 model is, if anything, a hair better. That's the point: selection isn't buying you a better model here, it's buying you a smaller one.

---
## Real-world example: all 7 columns vs. the 3 that matter

The chart below puts the two models from the table above side by side: one trained on all 7 columns, one trained on only the 3 columns every method agreed were useful. Look at how close the bars are on both R² and dollar error — that closeness *is* the finding, not a rounding artifact.

In [ ]:
_compare_labels = ["All 7 columns", "Top 3 (useful only)"]
_compare_r2 = [_r2_all, cross_val_score(Ridge(), _X[['dist_midtown', 'is_entire_home', 'is_manhattan']], _y, cv=_kf, scoring='r2').mean()]
_ridge_top3 = Ridge().fit(_Xtr[['dist_midtown', 'is_entire_home', 'is_manhattan']], _ytr)
_med_top3 = float(np.median(np.abs(_yva_price - np.expm1(_ridge_top3.predict(_Xva[['dist_midtown', 'is_entire_home', 'is_manhattan']])))))
_compare_med = [_med_all, _med_top3]

fig = go.Figure()
fig.add_trace(go.Bar(
    x=_compare_labels, y=_compare_r2, marker_color=[PALETTE["muted"], PALETTE["primary"]],
    text=[f"{v:.4f}" for v in _compare_r2], textposition="outside",
))
layout = base_layout(
    title="Cross-validated R²: all columns vs. the 3 useful ones",
    xaxis_title="", yaxis_title="5-fold CV R²",
)
layout.update(yaxis=dict(range=[0, 0.45]))
go.Figure(data=fig.data, layout=layout).show()

print(f"All 7 columns:      CV R2 = {_compare_r2[0]:.4f}   typical listing error = ${_compare_med[0]:.2f}")
print(f"Top 3 useful only:  CV R2 = {_compare_r2[1]:.4f}   typical listing error = ${_compare_med[1]:.2f}")


> **Discussion question:** `calculated_host_listings_count` — how many listings a host manages — sounds like it should matter for price. Its measured correlation with price is 0.0002. What real-world reasons could explain why a column that sounds meaningful turns out to carry no signal?

---
## Key takeaway

> **Feature selection rarely makes a model more accurate on its own — its real payoff is finding out that most of your columns weren't doing anything, and dropping them for free.**

---
*Next up: 08_dimensionality_reduction_intro — a light introduction to PCA before the full treatment in unsupervised/*